In [1]:
import os
import glob
import re
import time
import code
import numpy as np
import pandas as pd
import anndata
import pyfaidx
import pybedtools
from pybedtools import BedTool
import anndata as ad
import scipy.sparse as sp_sparse
import pickle
import scanpy as sc
import muon.atac as ma  # 用于 TF-IDF
import networkx as nx
from matplotlib import rcParams
import sys
sys.path.append('/home/liyang/BioWuYan/MethodTest/dygmamba/')


# Configuration

In [2]:
data_path = '/home/liyang/BioWuYan/BioProject/Final_Data/GM12878/'

output_path = "/home/liyang/BioWuYan/MethodTest/Data/All/"

# ATAC read

## From h5ad

In [3]:
adata_atac = ad.read_h5ad(data_path + "original_adata_atac_v1.h5ad")

In [ ]:
adata_atac.write_h5ad(output_path + "atac_origin.h5ad")

print("ATAC data saved to", output_path + "atac_origin.h5ad")

## From txt

In [5]:

print("--- 步骤 1: 加载计数矩阵 ---")

counts_file = '/home/liyang/BioWuYan/BioProject/Final_Data/GM12878/' + "atac_bams/atac_counts_matrix.txt"

# 使用 pandas 加载，这可能会很慢

counts_data = pd.read_csv(counts_file, header=None, sep="\t")

print("--- 步骤 2: 清理矩阵 ---")

# --- 1. 创建 Peak 名称 (行名) 和 Peak 信息 (用于 .var) ---
# (我们使用第1, 2, 3列 (chr, start, end))

peak_info = counts_data.iloc[:, [0, 1, 2]].copy()
peak_info.columns = ['chr', 'start', 'end']
# 创建 chr-start-end 格式的行名

row_names = peak_info['chr'] + "-" + peak_info['start'].astype(str) + "-" + peak_info['end'].astype(str)
peak_info.index = row_names

start_col_index = 10 

print(f"将从第 {start_col_index + 1} 列开始提取计数...")
counts_matrix_df = counts_data.iloc[:, start_col_index:]
counts_matrix_df.index = row_names # 设置行名 (peaks)

# --- 3. 创建细胞名称 (列名) ---
# 按字母顺序排序，以匹配 bedtools multicov (它会按 -bams 传入的顺序)

bam_files = sorted([f for f in os.listdir(data_path+"atac_bams") 
                    if f.startswith("atac_") and f.endswith(".bam")])

cell_names = [f.replace("atac_", "").replace(".bam", "") for f in bam_files]

counts_matrix_df.columns = cell_names

# print(counts_matrix_df)


print("--- 步骤 3: 创建 anndata 对象 ---")
# anndata 期望 (obs x var)，即 (cells x peaks)

adata_atac = ad.AnnData(counts_matrix_df.T)

# adata_atac.var_names = counts_matrix_df.index

# 将 peak 坐标信息存储在 .var 中
adata_atac.var = peak_info
print(f"创建 anndata 对象，维度: {adata_atac.shape}") # (n_cells, n_peaks)

from scipy import sparse

adata_atac.X = sparse.csr_matrix(adata_atac.X)


--- 步骤 1: 加载计数矩阵 ---
--- 步骤 2: 清理矩阵 ---
将从第 11 列开始提取计数...
--- 步骤 3: 创建 anndata 对象 ---
创建 anndata 对象，维度: (500, 72569)


In [6]:
new_cell_names = ['cell_'+str(i) for i in range(1, len(adata_atac.obs_names)+1)]

adata_atac = adata_atac[new_cell_names,:]

adata_atac.write_h5ad(output_path + "atac_origin_txt.h5ad")

print("文件已保存为:", output_path + "atac_origin_txt.h5ad")

/home/liyang/BioWuYan/conda_env/singlecellpreprocess/lib/python3.9/site-packages/anndata/_core/anndata.py:1209: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  df[key] = c
... storing 'chr' as categorical


文件已保存为: /home/liyang/BioWuYan/MethodTest/Data/All/atac_origin_txt.h5ad


# RNA read

## From h5ad

In [ ]:
adata_rna = ad.read_h5ad(data_path + "original_adata_rna.h5ad")

new_cell_names = ['cell_'+str(i) for i in range(1, len(adata_rna.obs_names)+1)]

adata_rna = adata_rna[new_cell_names,:]


In [ ]:
adata_rna.write_h5ad(output_path + "rna_origin.h5ad")

print("RNA data saved to", output_path + "rna_origin.h5ad")

## From txt

In [3]:
from src.data_read import create_anndata_from_folder

# get gene counts

print("********************---read gene & atac counts---*****************************")

rna_counts_directory =  '/home/liyang/BioWuYan/BioProject/Final_Data/GM12878/'+ 'cells_gene_count/'

adata_rna = create_anndata_from_folder(rna_counts_directory, type= 'gene')

********************---read gene & atac counts---*****************************


Processing files: 100%|██████████| 500/500 [08:44<00:00,  1.05s/it]


Create AnnData Object...
Successfully Create AnnData!


In [4]:
adata_rna.write_h5ad( output_path+ "rna_origin_txt.h5ad")

print("RNA txt data saved to", output_path + "rna_origin_txt.h5ad")

RNA txt data saved to /home/liyang/BioWuYan/MethodTest/Data/All/rna_origin_txt.h5ad


# Gene info

In [ ]:
from src.data_read import read_genenotation

print("********************---read gene info---*****************************")

gtf_file_path = "/home/liyang/BioWuYan/MethodTest/Data/All/annotation/gencode.V49.annotation.gtf" 

gtf_df = read_genenotation(gtf_file_path)

gtf_df.to_pickle(output_path + "gene_info_data.pkl")

print(gtf_df)

print("Successfully svae Gene info data to", output_path + "gene_info_data.pkl")

# ChIP_seq data

## Self ChIP-seq

In [ ]:
from src.data_read import read_chipseq

tf_chip_seq_path = "/home/liyang/BioWuYan/MethodTest/Data/All/TF_ChIP_seq"

tf_chip_seq_data = read_chipseq(tf_chip_seq_path)

tf_chip_seq_data.to_pickle(output_path + "tf_chip_seq_data.pkl")

print(tf_chip_seq_data)

print("Successfully svae ChIP-seq data to", output_path + "tf_chip_seq_data.pkl")

## SCENIC+ ChIP-seq

### ChIP-seq for cell line

In [3]:
import pandas as pd
from src.data_read import read_IDR_peaks_TFs

# ================= 配置 =================
root_dir = "."
output_file = "combined_tf_peaks_robust.parquet"

tf_chip_seq_scenic = read_IDR_peaks_TFs(root_dir, output_file)




In [1]:
import pandas as pd

# 读取整个文件
tf_chip_seq_scenic = pd.read_parquet("/home/liyang/BioWuYan/MethodTest/Data/All/SCENIC_plus/combined_tf_peaks_robust.parquet")

In [3]:
tf_chip_seq_scenic.columns

Index(['chrom', 'start', 'end', 'name', 'score', 'strand', 'signal_value',
       'p_value', 'q_value', 'peak', '10', '11', '12', '13', '14', '15', '16',
       '17', '18', '19', 'tf_name', 'cell_type', 'source_file_id'],
      dtype='object')

### Unibind data

In [5]:
import pandas as pd
from src.data_read import read_unibind_file

# ================= 配置 =================

data_tmp_path = "/home/liyang/BioWuYan/MethodTest/Data/All/Unibind/"
input_file = "hg38_compressed_TFBSs.bed.gz"
output_file = "unibind_hg38_cleaned.parquet"


# =======================================
df = read_unibind_file(data_tmp_path + input_file)

# --- 保存 ---
print(f"\n3. 正在保存为 {output_file} ...")

# 优化：只保存有用的列，丢弃 thickStart, thickEnd, itemRgb 这些绘图用的列
final_cols = ['chrom', 'start', 'end', 'tf_name', 'cell_type', 'score', 'strand', 'name']

df[final_cols].to_parquet(data_tmp_path + output_file, index=False)

print("✅ 全部完成！")

1. 开始读取数据...
读取完成，共 97492844 行。
前 5 行预览（原始数据）：
  chrom  start                                               name
0  chr1  10143  EXP040021_K562--myelogenous-leukemia-_NR2F6_MA...
1  chr1  10143  EXP040021_K562--myelogenous-leukemia-_NR2F6_MA...
2  chr1  10175  EXP000883_T47D--invasive-ductal-carcinoma-_ESR...
3  chr1  15626      EXP040282_foreskin-keratinocyte_CTCF_MA0139.1
4  chr1  16243  EXP000597_HeLa-S3--cervical-adenocarcinoma-_CT...


In [13]:
import pandas as pd

# 读取整个文件
tf_chip_seq_unibind = pd.read_parquet("/home/liyang/BioWuYan/MethodTest/Data/All/Unibind/unibind_hg38_cleaned.parquet")

In [14]:
tf_chip_seq_unibind

,chrom,start,end,tf_name,cell_type,score,strand,name
0,chr1,10143,10157,NR2F6,K562 myelogenous leukemia,0,-,EXP040021_K562--myelogenous-leukemia-_NR2F6_MA...
1,chr1,10143,10158,NR2F6,K562 myelogenous leukemia,0,-,EXP040021_K562--myelogenous-leukemia-_NR2F6_MA...
2,chr1,10175,10192,ESR1,T47D invasive ductal carcinoma,0,-,EXP000883_T47D--invasive-ductal-carcinoma-_ESR...
3,chr1,15626,15645,CTCF,foreskin keratinocyte,0,-,EXP040282_foreskin-keratinocyte_CTCF_MA0139.1
4,chr1,16243,16262,CTCF,HeLa S3 cervical adenocarcinoma,0,+,EXP000597_HeLa-S3--cervical-adenocarcinoma-_CT...
...,...,...,...,...,...,...,...,...
97492839,chrY,56883883,56883894,GATA2,lncap csfcs,0,+,GSE69043_lncap-csfcs_GATA2_MA0036.3
97492840,chrY,56883884,56883892,GATA3,MCF7 Invasive ductal breast carcinoma,0,-,EXP039573_MCF7--Invasive-ductal-breast-carcino...
97492841,chrY,56883884,56883892,GATA3,T47D invasive ductal carcinoma,0,-,EXP049899_T47D--invasive-ductal-carcinoma-_GAT...
97492842,chrY,56885229,56885241,CDX2,Wnt Fgf specified intestinal culture cells,0,+,EXP058137_Wnt-Fgf-specified-intestinal-culture...


# JASPAR data

## from txt

In [ ]:
from src.data_read import read_jaspar_folder

jaspar_folder_path = "/home/liyang/BioWuYan/MethodTest/Data/All/jaspar/"

genome_fasta_file = "/home/liyang/BioWuYan/MethodTest/Data/All/hg38.fa"

jaspar_data = read_jaspar_folder(jaspar_folder_path, genome_fasta_file, adata_atac)

jaspar_data.write_h5ad(output_path + "jaspar_data.h5ad")



## from glue

In [ ]:

jaspar_path = "/home/liyang/BioWuYan/MethodTest/Data/All/JASPAR2022-hg38.bed.gz"

from src.data_read import read_jaspar_bed

jaspar_data = read_jaspar_bed(jaspar_path)

jaspar_data.to_pickle(output_path + "jaspar_data.pkl")

print(jaspar_data)

print("Successfully svae JASPAR data to", output_path + "jaspar_data.pkl")


# Hi-C data

In [ ]:
from src.data_read import read_hic_data

print("********************---read hic---*****************************")

HIC_FILE = "/home/liyang/BioWuYan/MethodTest/Data/All/hic/GM12878.hic"

RESOLUTION = 10000
NORMALIZATION = 'VC'

hic_data_df = read_hic_data(
    hic_file_path=HIC_FILE,
    resolution=RESOLUTION,
    normalization=NORMALIZATION
)

hic_data_df.to_pickle(output_path + 'hic_data.pkl')
